# LAB D7 - Regresión lineal con SQLite: Abarrotes Don Carlos

**Curso:** Fundamentos de Gestión de Datos  
**Semana:** 7  
**Objetivo:** entrenar una regresión lineal usando la base SQLite real descargada desde GitHub, sin generar datos aleatorios.


<!-- BLOQUE_AGREGADO_SUPUESTOS_SQL_D7 -->
## Supuestos e hipótesis del modelo SQLite

**Modelo evalúado:** regresión lineal para predecir `monto_venta_soles` a partir de variables reales de la base SQLite `abarrotes_ventas_inventario.db`.

**Hipótesis general del modelo**

| Elemento | Planteamiento |
|---|---|
| H0 | Las variables selecciónadas no explican de forma relevante el monto de venta. En términos prácticos, el modelo no mejora frente a predecir solo el promedio. |
| H1 | Al menos una variable predictora aporta información para explicar o predecir el monto de venta. El modelo mejora frente a usar solo el promedio. |
| Criterio de decisión | Se rechaza H0 si el desempeño es alto en prueba, especialmente si `R² test` es elevado y el `RMSE` es razonable para el negocio. |

**Supuestos que se deben verificar**

| Supuesto | Cómo se revisa en el notebook | Criterio práctico |
|---|---|---|
| Linealidad | Gráfico de ventas reales vs predichas y patrón de residuos | Los puntos deben acercarse a la línea de predicción perfecta; los residuos no deben mostrar curva clara. |
| Independencia de errores | Durbin-Watson sobre residuos | Valores cercanos a 2 sugieren independencia razonable. |
| Homocedasticidad | Breusch-Pagan y residuos vs predichos | Si p-valor > 0.05, no hay evidencia fuerte de varianza desigual. |
| Normalidad de residuos | Shapiro-Wilk e histograma | Si p-valor > 0.05, los residuos son compatibles con normalidad; si no, se reporta como limitación. |
| Variables reales y sin datos inventados | Carga directa desde GitHub/SQLite | La base debe provenir de `abarrotes_ventas_inventario.db`, no de datos generados aleatoriamente. |


## Paso 1 - Librerías y descarga de la base SQLite

In [ ]:
import sqlite3
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DB_URL = (
    "https://raw.githubusercontent.com/Rociosayan/"
    "PMD1_Fundamentos_Gestion_Datos/main/casos/"
    "16_abarrotes_ventas_inventario/abarrotes_ventas_inventario.db"
)
DB_PATH = Path("abarrotes_ventas_inventario.db")

urllib.request.urlretrieve(DB_URL, DB_PATH)
conn = sqlite3.connect(DB_PATH)
print("Base de datos de Abarrotes Don Carlos cargada correctamente.")


## Paso 2 - Verificación de tablas y carga desde SQLite

In [ ]:
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
display(tablas)

df_raw = pd.read_sql_query("SELECT * FROM ventas_original", conn)
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
display(df_raw.head())
display(df_raw.dtypes.to_frame("tipo"))


## Paso 3 - Limpieza de datos reales

La tabla viene desnormalizada y varias columnas numéricas están guardadas como texto. Por eso se limpian antes del modelo.

In [ ]:
def limpiar_numero(serie):
    extraido = serie.astype(str).str.extract(r"(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(extraido, errors="coerce")

def limpiar_texto(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"nan": np.nan, "none": np.nan})
    )

def parsear_fecha_mixta(serie):
    texto = serie.astype(str).str.strip()
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    es_iso = texto.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    resultado.loc[es_iso] = pd.to_datetime(texto.loc[es_iso], format="%Y-%m-%d", errors="coerce")
    resultado.loc[~es_iso] = pd.to_datetime(texto.loc[~es_iso], errors="coerce", dayfirst=True)
    return resultado

df = df_raw.copy()
columnas_numericas_base = [
    "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
    "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
    "monto_venta_soles", "margen_venta_soles",
    "productos_costo_unitario", "productos_precio_lista",
]
for columna in columnas_numericas_base:
    df[columna] = limpiar_numero(df[columna])

columnas_categoricas = [
    "canal", "metodo_pago", "tiendas_zona", "tiendas_tipo_local",
    "productos_categoria", "clientes_segmento",
]
for columna in columnas_categoricas:
    df[columna] = limpiar_texto(df[columna])

df["fecha_operacion_dt"] = parsear_fecha_mixta(df["fecha_operacion"])
df["mes_operacion"] = df["fecha_operacion_dt"].dt.month
df["dia_semana"] = df["fecha_operacion_dt"].dt.dayofweek
df["fin_semana"] = df["dia_semana"].isin([5, 6]).astype(int)
df["importe_bruto_soles"] = df["cantidad_unidades"] * df["precio_venta_unitario"]
df["importe_con_descuento_soles"] = df["importe_bruto_soles"] * (1 - df["descuento_pct"].fillna(0) / 100)
df["utilidad_unitaria_soles"] = df["precio_venta_unitario"] - df["costo_unitario"]

df = df.dropna(subset=["monto_venta_soles"]).copy()
print(f"Datos limpios para modelar: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df[["id_venta", "cantidad_unidades", "precio_venta_unitario", "importe_bruto_soles", "monto_venta_soles"]].head())


## Paso 4 - Preparación de variables y division train/test

In [ ]:
columnas_numericas = [
    "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
    "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
    "productos_costo_unitario", "productos_precio_lista",
    "importe_bruto_soles", "importe_con_descuento_soles", "utilidad_unitaria_soles",
    "mes_operacion", "dia_semana", "fin_semana",
]
target = "monto_venta_soles"

X = df[columnas_numericas + columnas_categoricas]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Train: {X_train.shape[0]} registros")
print(f"Test : {X_test.shape[0]} registros")
print("Target:", target)
print("Variables numericas:", columnas_numericas)
print("Variables categoricas:", columnas_categoricas)


## Paso 5 - Entrenamiento de LinearRegression

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocesador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), columnas_numericas),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", encoder),
        ]), columnas_categoricas),
    ]
)

modelo = Pipeline([
    ("preprocesamiento", preprocesador),
    ("modelo", LinearRegression()),
])

modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")


## Paso 6 - Evalúación del modelo

In [ ]:
pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

r2_train = r2_score(y_train, pred_train)
r2_test = r2_score(y_test, pred_test)
mse_test = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)

print(f"R2 train : {r2_train:.4f}")
print(f"R2 test  : {r2_test:.4f}")
print(f"MSE test : {mse_test:.2f}")
print(f"RMSE test: S/ {rmse_test:.2f}")

# Interpretacion rapida
if r2_test >= 0.80:
    print("El ajuste es alto: el modelo explica gran parte de la variacion del monto de venta.")
elif r2_test >= 0.50:
    print("El ajuste es moderado: el modelo captura parte importante del patron, pero puede mejorar.")
else:
    print("El ajuste es bajo: faltan variables o relaciones no lineales relevantes.")


In [ ]:
# BLOQUE_AGREGADO_DIAGNOSTICO_SQL_D7
# Verificación de supuestos e hipótesis del modelo SQL/SQLite
import sys
import subprocess

try:
    from scipy import stats
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import het_breuschpagan
    from statsmodels.stats.stattools import durbin_watson
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import het_breuschpagan
    from statsmodels.stats.stattools import durbin_watson

residuos_test = y_test.values - pred_test
shapiro_stat, shapiro_p = stats.shapiro(residuos_test)
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(residuos_test, sm.add_constant(pred_test))
dw = durbin_watson(residuos_test)
corr_resid_pred = np.corrcoef(pred_test, residuos_test)[0, 1]

resumen_supuestos_sql = pd.DataFrame([
    {
        "supuesto": "Hipótesis general del modelo",
        "prueba_o_indicador": "R2 test y RMSE",
        "resultado": f"R2 test={r2_test:.4f}; RMSE=S/ {rmse_test:.2f}",
        "decision": "Se rechaza H0" if r2_test >= 0.80 else "No se rechaza H0 con fuerza suficiente",
    },
    {
        "supuesto": "Normalidad de residuos",
        "prueba_o_indicador": "Shapiro-Wilk",
        "resultado": f"p-valor={shapiro_p:.6f}",
        "decision": "No se rechaza normalidad" if shapiro_p > 0.05 else "Se rechaza normalidad; reportar como limitación",
    },
    {
        "supuesto": "Homocedasticidad",
        "prueba_o_indicador": "Breusch-Pagan",
        "resultado": f"p-valor={bp_f_p:.6f}",
        "decision": "No hay evidencia fuerte de heterocedasticidad" if bp_f_p > 0.05 else "Hay evidencia de heterocedasticidad",
    },
    {
        "supuesto": "Independencia de errores",
        "prueba_o_indicador": "Durbin-Watson",
        "resultado": f"DW={dw:.4f}",
        "decision": "Independencia razonable" if 1.5 <= dw <= 2.5 else "Revisar autocorrelación",
    },
    {
        "supuesto": "Linealidad práctica",
        "prueba_o_indicador": "Correlación predicho-residuo",
        "resultado": f"corr={corr_resid_pred:.4f}",
        "decision": "Sin patrón lineal fuerte en residuos" if abs(corr_resid_pred) < 0.20 else "Revisar patrón de residuos",
    },
])

display(resumen_supuestos_sql)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(pred_test, residuos_test, alpha=0.75, edgecolor="white")
plt.axhline(0, color="red", linestyle="--")
plt.title("Supuesto: residuos vs predichos")
plt.xlabel("Monto predicho (S/)")
plt.ylabel("Residuo")

plt.subplot(1, 2, 2)
plt.hist(residuos_test, bins=18, edgecolor="black", alpha=0.80)
plt.axvline(0, color="red", linestyle="--")
plt.title("Supuesto: normalidad de residuos")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


<!-- BLOQUE_AGREGADO_INTERPRETACION_SQL_D7 -->
### Decisión sobre la hipótesis en SQL/SQLite

Con el modelo de regresión lineal aplicado a la base real de abarrotes, el `R² test` esperado es alto. Por ello, se **rechaza H0** y se acepta que las variables selecciónadas s? explican una parte importante del monto de venta. La decisión se apoya principalmente en `importe_bruto_soles`, `cantidad_unidades`, `precio_venta_unitario` y variables categóricas del negocio.

La revisión de supuestos no busca que todo sea perfecto, sino documentar si el modelo es defendible. Si normalidad u homocedasticidad no se cumplen completamente, se reporta como limitación y se recomienda probar transformaciones o modelos no lineales en trabajos posteriores.


## Paso 7 - Coeficientes e interpretación

In [ ]:
nombres = modelo.named_steps["preprocesamiento"].get_feature_names_out()
nombres = [n.replace("num__", "").replace("cat__", "") for n in nombres]
coeficientes = (
    pd.DataFrame({
        "variable": nombres,
        "coeficiente": modelo.named_steps["modelo"].coef_,
    })
    .assign(abs_coeficiente=lambda d: d["coeficiente"].abs())
    .sort_values("abs_coeficiente", ascending=False)
    .drop(columns="abs_coeficiente")
)

display(coeficientes.head(15))


## Paso 8 - Gráfico de ventas reales vs predichas

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, pred_test, alpha=0.75, edgecolor="white")
minimo = min(y_test.min(), pred_test.min())
maximo = max(y_test.max(), pred_test.max())
plt.plot([minimo, maximo], [minimo, maximo], "r--", label="Prediccion perfecta")
plt.title(f"Regresion lineal: ventas reales vs predichas\nR2 test = {r2_test:.3f}")
plt.xlabel("Monto real de venta (S/)")
plt.ylabel("Monto predicho de venta (S/)")
plt.legend()
plt.tight_layout()
plt.show()


## Paso 9 - Guardar resultados en SQLite

In [ ]:
resultados = X_test.copy()
resultados.insert(0, "id_venta", df.loc[X_test.index, "id_venta"].values)
resultados["monto_real_soles"] = y_test.values
resultados["monto_predicho_soles"] = np.round(pred_test, 2)
resultados["residuo_soles"] = np.round(y_test.values - pred_test, 2)
resultados["error_abs_soles"] = resultados["residuo_soles"].abs()

df_limpio = df[["id_venta", "fecha_operacion", target] + columnas_numericas + columnas_categoricas].copy()
df_limpio.to_sql("ventas_limpias_regresion", conn, if_exists="replace", index=False)
resultados.to_sql("predicciones_regresion_lineal", conn, if_exists="replace", index=False)
conn.commit()

print("Tablas guardadas:")
print("- ventas_limpias_regresion")
print("- predicciones_regresion_lineal")

display(pd.read_sql_query("SELECT * FROM predicciones_regresion_lineal LIMIT 10", conn))


## Respuestás sugeridas

**Target:** `monto_venta_soles`, porque representa el valor monetario de cada venta.  
**Variables más importantes:** normalmente `importe_bruto_soles`, `cantidad_unidades` y `precio_venta_unitario`, porque definen directamente el monto vendido.  
**Interpretación:** si el R² test es alto, el modelo explica bien el comportamiento del monto de venta con datos reales de la base SQLite.  
**Limitacion:** la regresión lineal no captura relaciónes complejas si no se agregan variables derivadas como `importe_bruto_soles`.
